# Analysis companion for the separate Quarto paper

This is **Pathway 2**. The notebook produces and audits the evidence. The paper in
this folder carries the complete research narrative. Both pathways use the same
root-level data, code, figures, and frozen conclusion.


In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import Image, display

HERE = Path.cwd()
candidates = [HERE, HERE.parent, HERE / "project" / "example_real_sec",
              Path("/content/uc_finmlai/project/example_real_sec")]
ROOT = next((p for p in candidates if (p / "analysis.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run from project/example_real_sec/pathway2_separate_paper or the project root.")
sys.path.insert(0, str(ROOT))
import analysis as A
frame = pd.read_csv(ROOT / "data/derived/scored_events.csv", dtype={"cik": str})
summary = json.loads((ROOT / "data/derived/summary.json").read_text())
print(f"Project root: {ROOT}")
print(f"Rows: {len(frame)}; later test: {(frame.split == 'test').sum()}")


## Audit one source record

The feature is Apple's selected Exhibit 99.1, accepted with the February 2, 2023
8-K. The label is next-quarter revenue growth computed from a later filing.


In [ ]:
cols = ["ticker", "accession_number", "decision_time_utc", "exhibit_url",
        "release_sha256", "outcome_available_date", "outcome_accession_number"]
display(frame.loc[frame.accession_number == "0000320193-23-000005", cols].T)
display(Image(filename=str(ROOT / "figures/01_information_clock.png")))


## Recreate the declared evidence

`analysis.py` reads the committed score ledger, recomputes the later-test metrics,
and rebuilds every result figure. Add `--score-finbert` only when intentionally
re-scoring the raw releases with the pinned checkpoint.


In [ ]:
A.run()
display(Image(filename=str(ROOT / "figures/03_model_comparison.png")))
display(Image(filename=str(ROOT / "figures/04_review_policy.png")))


In [ ]:
print("Lexicon test rank IC:", round(summary["test"]["lexicon_ic"], 3))
print("FinBERT test rank IC:", round(summary["test"]["finbert_ic"], 3))
print("Paired difference:", round(summary["test"]["finbert_minus_lexicon_ic"], 3))
print("Paired 90% interval:", [round(x, 3) for x in summary["test"]["finbert_minus_lexicon_ic_90"]])


## Hand the evidence to the paper

The Quarto paper references the same root-level figures and reports the same
metrics. It may improve explanation, but it may not change the model, population,
threshold, or final test after seeing these results.
